# Private AI Services Container (PASC) Integration Quickstart

Experience the Private AI Services Container + Oracle VecDB flow end-to-end: extract documentation text, create embeddings, and store/search vectors in a BYOV table.

**What you'll do**

1. Setup dependencies
2. Define Private AI Services Container embeddings client
3. Configure VecDB + Private AI Services Container connections
4. Create BYOV vector table
5. Build documentation extraction and chunking utilities
6. Ingest docs (embed + upsert)
7. Query for answers
8. Optional cleanup


## 1. Setup

Install the required Python packages:

```bash
pip install requests python-dotenv oracle-vecdb beautifulsoup4 langchain-text-splitters
```


In [ ]:
%pip install requests python-dotenv oracle-vecdb beautifulsoup4 langchain-text-splitters


## 2. Define Private AI Services Container Embeddings Client

Create a lightweight client for calling the Private AI Services Container embeddings endpoint.


In [ ]:
import json
import os

import requests


class PrivateAIServiceContainer:
    """Client for the embeddings service."""
    def __init__(self, base_url: str, api_key: str, cert_path: str, model_name: str):
        self.base_url = base_url
        self.api_key = api_key
        self.cert_path = cert_path
        self.model_name = model_name

    def generate_embeddings(self, text_list):
        """Input: List[str], Output: List[List[float]]"""
        headers = {
            'Authorization': f'Bearer {self.api_key}',
            'Content-Type': 'application/json',
        }
        payload = {
            'model': self.model_name,
            'input': text_list,
        }

        resp = requests.post(
            f"{self.base_url}/v1/embeddings",
            headers=headers,
            data=json.dumps(payload),
            verify=self.cert_path,
            proxies={"http": None, "https": None},
        )

        if resp.status_code != 200:
            raise Exception(f"Embedding failed: {resp.text}")

        result = resp.json()
        return [item['embedding'] for item in result['data']]


## 3. Configure VecDB + Private AI Services Container Connections

Load environment variables, initialize the Oracle VecDB client, and configure the PASC client settings.


In [ ]:
import os

from dotenv import find_dotenv
print(find_dotenv())

from dotenv import load_dotenv
from oracle_vecdb import OracleVecDB, Configuration

load_dotenv()

# VecDB connection info 
vecdb_host = os.getenv('VECDB_REST_URL')
vecdb_user = os.getenv('VECDB_USERNAME') or os.getenv('VECDB_USER')
vecdb_password = os.getenv('VECDB_PASSWORD')
vecdb_access_token = os.getenv('VECDB_ACCESS_TOKEN')


# Build VecDB client config
vecdb_config_kwargs = {"rest_url": vecdb_host}
if vecdb_access_token:
    vecdb_config_kwargs["access_token"] = vecdb_access_token
else:
    vecdb_config_kwargs["username"] = vecdb_user
    vecdb_config_kwargs["password"] = vecdb_password
vecdb_config = Configuration(**vecdb_config_kwargs)
if os.getenv('VECDB_SELF_SIGNED_SSL', 'false').lower() == 'true':
    vecdb_config.verify_ssl = False
    import urllib3
    urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
    print('VecDB SSL verification disabled (self-signed cert)')

vecdb_client = OracleVecDB(vecdb_config)
auth_method = 'bearer token' if vecdb_config.access_token else 'username/password'
print(f"SDK client ready, using REST endpoint: {vecdb_config.rest_url}")
print('Auth method:', auth_method)


PASC_BASE_URL = os.getenv('PASC_HOST')
PASC_API_KEY = os.getenv('PASC_API_KEY')
PASC_CERT = os.getenv('PASC_CERT')
PASC_MODEL = os.getenv('PASC_MODEL')


pasc_client = PrivateAIServiceContainer(
    base_url=PASC_BASE_URL,
    api_key=PASC_API_KEY,
    cert_path=PASC_CERT,
    model_name=PASC_MODEL,
)
print(f"Private AI Services Container client ready, using host: {PASC_BASE_URL}")

table_name = os.getenv("PASC_TABLE") or "pasc_vectors_byov"
print("Target table:", table_name)


## 4. Create BYOV Vector Table

Create a BYOV vector table with manual indexing to store Private AI Services Container embeddings.


In [ ]:
# Create BYOV vector table when it is not already present.
# table_name is set in the configuration cell above.
try:
    vecdb_client.describe_vector_table(name=table_name)
    print(f"Table {table_name} already exists; using existing schema")
except Exception:
    # Do not pass vector_index_params here. In this environment, minimal
    # vector index params such as {"auto_index": True/False} can trigger ORA-57715.
    response = vecdb_client.create_vector_table(
        name=table_name,
        comment="PASC BYOV vectors",
        table_params={"auto_generate_id": False},
        annotations={
            "source": "string",
            "chunk_index": "int",
            "content": "string",
        },
    )
    print(f"{response}")


## 5. Documentation Extraction and Chunking Utilities

Define helpers to extract text from a web documentation page and split it into optimized chunks for embedding.


In [ ]:
import os
import socket
import time
import requests
from bs4 import BeautifulSoup


HOST = socket.gethostname()

# Optional proxies (prefer .env). If provided, normalize to both cases.
http_proxy = os.getenv('HTTP_PROXY') or os.getenv('http_proxy')
https_proxy = os.getenv('HTTPS_PROXY') or os.getenv('https_proxy')
if http_proxy:
    os.environ.setdefault('HTTP_PROXY', http_proxy)
    os.environ.setdefault('http_proxy', http_proxy)
if https_proxy:
    os.environ.setdefault('HTTPS_PROXY', https_proxy)
    os.environ.setdefault('https_proxy', https_proxy)

# Ensure NO_PROXY includes localhost and the current host.
no_proxy = os.getenv('NO_PROXY') or os.getenv('no_proxy')
if not no_proxy:
    no_proxy = f'localhost,127.0.0.1,{HOST}'
    os.environ.setdefault('NO_PROXY', no_proxy)
    os.environ.setdefault('no_proxy', no_proxy)



In [ ]:
import requests
from bs4 import BeautifulSoup

URL = "https://docs.oracle.com/en-us/iaas/Content/generative-ai/overview.htm"

def load_paragraph_chunks(url: str) -> list[str]:
    html = requests.get(url, timeout=30).text
    soup = BeautifulSoup(html, "html.parser")

    chunks = []
    seen = set()

    for p in soup.find_all("p"):
        text = " ".join(p.get_text(" ", strip=True).split())

        if len(text) < 40:
            continue
        if text in seen:
            continue

        seen.add(text)
        chunks.append(text)

    return chunks
chunks = load_paragraph_chunks(URL)

print(f"chunk_count={len(chunks)}")
for idx, chunk in enumerate(chunks):
    print("---")
    print(f"paragraph_index={idx}")
    print(chunk)


## 6. Ingest Documentation: Embed and Upsert

Generate embeddings for each chunk and upsert the vectors into the BYOV table.


In [ ]:
from uuid import NAMESPACE_URL, uuid5

doc_urls = [
    URL
]

for doc_url in doc_urls:
    print(f'Ingesting {doc_url}')

    chunks = load_paragraph_chunks(doc_url)
    print(f'Generated {len(chunks)} chunks for doc: {doc_url}')

    embeddings = pasc_client.generate_embeddings(chunks)
    print(f'Generated {len(embeddings)} embeddings using PASC client')

    vectors = []
    for idx, (chunk, embedding) in enumerate(zip(chunks, embeddings)):
        vector_id = str(uuid5(NAMESPACE_URL, f"{doc_url}#chunk={idx}"))
        vectors.append({
            'id': vector_id,
            'dense_vector': embedding,
            'metadata': {
                'source': doc_url,
                'chunk_index': idx,
                'content': chunk,
            },
        })

    print(f'Upserting {len(vectors)} vectors into {table_name}')
    response = vecdb_client.upsert_vectors(
        table_name=table_name,
        vectors=vectors,
    )
    print('Upsert complete.')


## 7. Query for Answers

Embed questions, search the vector table, and print the top matching chunks.


In [ ]:
def query_items(response):
    if isinstance(response, list):
        return response
    if isinstance(response, tuple):
        return list(response)
    return getattr(response, "items", None) or getattr(response, "matches", None) or []


def result_metadata(item):
    return item.get("metadata", {}) if isinstance(item, dict) else getattr(item, "metadata", {})


def result_distance(item):
    if isinstance(item, dict):
        return item.get("distance")
    return getattr(item, "distance", getattr(item, "score", None))


def result_id(item):
    return item.get("id") if isinstance(item, dict) else getattr(item, "id", None)


def result_vector(item):
    if isinstance(item, dict):
        return item.get("vector") or item.get("dense_vector")
    return getattr(item, "vector", getattr(item, "dense_vector", None))


def result_text(item):
    return item.get("text", "") if isinstance(item, dict) else getattr(item, "text", "")


questions = [
    ('What is the OCI Generative AI service and what does it provide?', 2)
]
for question, k in questions:
    print('\n' + '=' * 60)
    print(f'Question: {question}')
    # Embed the question using the same embedding source
    query_vector = pasc_client.generate_embeddings([question])[0]

    # Search BYOV vector table using the query vector
    results = vecdb_client.query(
        table_name=table_name,
        query_by={
            'vector': query_vector,
            'vector_field': 'dense_vector',
        },
        top_k=k,
        include_vectors=False,
    )

    # Return concatenated top-k chunks as a simple answer
    items = query_items(results)
    if not items:
        print('No result fetched')
    else:
        print('\n\n'.join(
            result_metadata(item).get('content', '') for item in items
        ))


## 8. Optional Cleanup

Drop the vector table when `DROP_TABLE_WHEN_DONE=true` is set.


In [ ]:
if os.getenv("DROP_TABLE_WHEN_DONE", "false").lower() == "true":
    vecdb_client.drop_vector_table(name=table_name)
    print(f'Dropped table {table_name}')
else:
    print('Cleanup skipped. Set DROP_TABLE_WHEN_DONE=true to enable table drop.')